# 03 - Expanding Window Model Lab (All Inline)

Goals:
- Expanding windows with 14-day step and 14-day forecast horizon
- Initial lookbacks: 30, 45, 60 days
- Same metrics as before + extra diagnostics
- Per-model combined prediction dumps across all sub-runs
- MLflow parent run per model + nested run per window
- Final expanded run metrics + average metrics logged

In [ ]:
# =========================
# CONFIG (single block)
# =========================

from pathlib import Path

ROOT = Path("..").resolve()

INPUT_MODE = "daily_csv"  # daily_csv | daily_parquet
DAILY_CSV_PATH = ROOT / "artifacts" / "data" / "daily.csv"
DAILY_PARQUET_PATH = ROOT / "artifacts" / "data" / "daily.parquet"

LOOKBACK_WINDOWS_DAYS = [30, 45, 60]
PREDICTION_WINDOW_DAYS = 14
EXPAND_STEP_DAYS = 14
INCLUDE_FORCED_FINAL_WINDOW = True

LAG_DAYS = [1, 2, 3, 7, 14]

OUTPUT_DIR = ROOT / "artifacts" / "expanding_backtest"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
SUMMARY_DIR = OUTPUT_DIR / "summary"

SELECTION_WEIGHT_AVG = 0.5

ENABLE_MLFLOW = True
TRACKING_URI = "sqlite:///../mlruns.db"
EXPERIMENT_NAME = "milk_models_expanding"

SHOW_PLOTS = True
TOP_MODELS_TO_PLOT = 5

print("Lookbacks:", LOOKBACK_WINDOWS_DAYS)
print("Horizon:", PREDICTION_WINDOW_DAYS)
print("Step:", EXPAND_STEP_DAYS)

In [ ]:
import warnings
import re
import json

import numpy as np
import pandas as pd
import cloudpickle
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except Exception:
    Prophet = None
    PROPHET_AVAILABLE = False

try:
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
    SKLEARN_AVAILABLE = True
except Exception:
    LinearRegression = Ridge = RandomForestRegressor = ExtraTreesRegressor = GradientBoostingRegressor = None
    SKLEARN_AVAILABLE = False

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception:
    LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False

try:
    import mlflow
    MLFLOW_AVAILABLE = True
except Exception:
    mlflow = None
    MLFLOW_AVAILABLE = False

warnings.filterwarnings("ignore")

print("PROPHET_AVAILABLE:", PROPHET_AVAILABLE)
print("SKLEARN_AVAILABLE:", SKLEARN_AVAILABLE)
print("XGBOOST_AVAILABLE:", XGBOOST_AVAILABLE)
print("LIGHTGBM_AVAILABLE:", LIGHTGBM_AVAILABLE)
print("MLFLOW_AVAILABLE:", MLFLOW_AVAILABLE)

In [ ]:
# =========================
# DATA LOAD + DAILY PREP
# =========================

if INPUT_MODE == "daily_csv":
    if not DAILY_CSV_PATH.exists():
        raise FileNotFoundError(f"Missing {DAILY_CSV_PATH}. Run data cleaning notebook first.")
    daily_df = pd.read_csv(DAILY_CSV_PATH)
    source_path = DAILY_CSV_PATH
elif INPUT_MODE == "daily_parquet":
    if not DAILY_PARQUET_PATH.exists():
        raise FileNotFoundError(f"Missing {DAILY_PARQUET_PATH}. Run data cleaning notebook first.")
    daily_df = pd.read_parquet(DAILY_PARQUET_PATH)
    source_path = DAILY_PARQUET_PATH
else:
    raise ValueError(f"Unknown INPUT_MODE: {INPUT_MODE}")

daily_df["date"] = pd.to_datetime(daily_df["date"], errors="coerce")
if "gallons" not in daily_df.columns:
    if "total_milk_oz" not in daily_df.columns:
        raise ValueError("Input needs gallons or total_milk_oz")
    daily_df["gallons"] = pd.to_numeric(daily_df["total_milk_oz"], errors="coerce") / 128.0

daily_df["gallons"] = pd.to_numeric(daily_df["gallons"], errors="coerce")
daily_df = daily_df.dropna(subset=["date", "gallons"]).sort_values("date").reset_index(drop=True)

daily_df = daily_df.set_index("date").asfreq("D")
daily_df["gallons"] = daily_df["gallons"].fillna(0.0)
daily_df = daily_df.reset_index()

print("Source:", source_path)
print("Rows:", len(daily_df))
print("Date range:", daily_df["date"].min(), "->", daily_df["date"].max())
print("Mean gallons:", round(float(daily_df["gallons"].mean()), 3))
daily_df.head()

In [ ]:
# =========================
# UTILS: metrics, features, expanding splits
# =========================

def sanitize_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", name)


def metrics_dict(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    err = actual - pred

    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))

    mask = actual != 0
    if np.any(mask):
        mape = float(np.mean(np.abs(err[mask] / actual[mask])) * 100.0)
    else:
        mape = float("nan")

    accuracy = float(100.0 - mape) if not np.isnan(mape) else float("nan")
    bias = float(np.mean(pred - actual))
    mean_actual = float(np.mean(actual)) if len(actual) else float("nan")
    error_pct = float((rmse / mean_actual) * 100.0) if (mean_actual and mean_actual != 0) else float("nan")
    error_std = float(np.std(err))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "Accuracy": accuracy,
        "Bias": bias,
        "ErrorPct": error_pct,
        "ErrorStd": error_std,
    }


def _roll_mean(vals, w):
    if len(vals) < w:
        return float(np.mean(vals)) if len(vals) else 0.0
    return float(np.mean(vals[-w:]))


def _roll_std(vals, w):
    if len(vals) < w:
        return float(np.std(vals)) if len(vals) else 0.0
    return float(np.std(vals[-w:]))


def build_feature_row(history_vals, current_date, lag_days):
    eps = 1e-9
    row = {}

    for lag in lag_days:
        row[f"lag_{lag}"] = float(history_vals[-lag])

    mean7 = _roll_mean(history_vals, 7)
    mean14 = _roll_mean(history_vals, 14)

    row["mean_7"] = mean7
    row["mean_14"] = mean14
    row["std_7"] = _roll_std(history_vals, 7)
    row["std_14"] = _roll_std(history_vals, 14)
    row["mean_7_vs_14_diff"] = mean7 - mean14
    row["mean_7_vs_14_ratio"] = float(mean7 / (mean14 + eps))
    row["mean_7_vs_14_pct"] = float(((mean7 - mean14) / (abs(mean14) + eps)) * 100.0)

    row["day_of_week"] = int(current_date.dayofweek)
    row["is_weekend"] = int(current_date.dayofweek >= 5)
    row["month"] = int(current_date.month)
    row["day_of_month"] = int(current_date.day)
    row["week_of_year"] = int(current_date.isocalendar().week)

    return row


def build_train_xy(train_dates, train_vals, lag_days):
    max_lag = max(lag_days)
    rows = []
    y = []

    for i in range(max_lag, len(train_vals)):
        feat = build_feature_row(train_vals[:i], train_dates[i], lag_days)
        rows.append(feat)
        y.append(float(train_vals[i]))

    if not rows:
        return pd.DataFrame(), np.array([])

    return pd.DataFrame(rows), np.asarray(y, dtype=float)


def recursive_predict_feature_model(estimator, train_dates, train_vals, horizon_days, lag_days):
    history = list(np.asarray(train_vals, dtype=float))
    preds = []
    last_date = pd.Timestamp(train_dates[-1])

    for _ in range(horizon_days):
        next_date = last_date + pd.Timedelta(days=1)
        feat = build_feature_row(history, next_date, lag_days)
        x_next = pd.DataFrame([feat])
        p = float(estimator.predict(x_next)[0])
        p = max(p, 0.0)
        preds.append(p)
        history.append(p)
        last_date = next_date

    return np.asarray(preds, dtype=float)


def generate_expanding_splits(n_rows, initial_train_days, horizon_days, step_days, include_forced_final=True):
    if n_rows <= initial_train_days + horizon_days:
        return []

    out = []
    train_end_idx = initial_train_days - 1

    while train_end_idx + horizon_days < n_rows:
        out.append({
            "train_start_idx": 0,
            "train_end_idx": train_end_idx,
            "test_start_idx": train_end_idx + 1,
            "test_end_idx": train_end_idx + horizon_days,
        })
        train_end_idx += step_days

    final_train_end = n_rows - horizon_days - 1
    if include_forced_final and final_train_end >= initial_train_days - 1:
        if not out or out[-1]["train_end_idx"] != final_train_end:
            out.append({
                "train_start_idx": 0,
                "train_end_idx": final_train_end,
                "test_start_idx": final_train_end + 1,
                "test_end_idx": final_train_end + horizon_days,
            })

    return out

In [ ]:
# =========================
# MODEL CATALOG
# =========================

model_catalog = {}

if SKLEARN_AVAILABLE:
    model_catalog["LinearRegression"] = {"family": "feature", "builder": lambda: LinearRegression()}
    model_catalog["Ridge"] = {"family": "feature", "builder": lambda: Ridge(alpha=1.0)}
    model_catalog["RandomForest"] = {
        "family": "feature",
        "builder": lambda: RandomForestRegressor(
            n_estimators=500,
            max_depth=8,
            random_state=42,
            n_jobs=-1,
        ),
    }
    model_catalog["ExtraTrees"] = {
        "family": "feature",
        "builder": lambda: ExtraTreesRegressor(
            n_estimators=500,
            max_depth=8,
            random_state=42,
            n_jobs=-1,
        ),
    }
    model_catalog["GradientBoosting"] = {
        "family": "feature",
        "builder": lambda: GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        ),
    }

if XGBOOST_AVAILABLE:
    model_catalog["XGBoost"] = {
        "family": "feature",
        "builder": lambda: XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            objective="reg:squarederror",
        ),
    }

if LIGHTGBM_AVAILABLE:
    model_catalog["LightGBM"] = {
        "family": "feature",
        "builder": lambda: LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            random_state=42,
        ),
    }

model_catalog["ARIMA_212"] = {"family": "arima", "order": (2, 1, 2)}
model_catalog["SARIMA_212x111_7"] = {
    "family": "sarima",
    "order": (2, 1, 2),
    "seasonal_order": (1, 1, 1, 7),
}

if PROPHET_AVAILABLE:
    model_catalog["Prophet"] = {
        "family": "prophet",
        "params": {
            "seasonality_mode": "multiplicative",
            "weekly_seasonality": True,
            "yearly_seasonality": False,
            "daily_seasonality": False,
            "changepoint_prior_scale": 0.05,
            "seasonality_prior_scale": 1.0,
        },
    }

print("Model count:", len(model_catalog))
print(sorted(model_catalog.keys()))

In [ ]:
# =========================
# BACKTEST + CUSTOM DUMPS
# =========================

for d in [OUTPUT_DIR, PREDICTIONS_DIR, METRICS_DIR, MODELS_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

dates = daily_df["date"].to_numpy()
values = daily_df["gallons"].astype(float).to_numpy()

all_metrics_rows = []
all_prediction_rows = []
artifact_manifest = {}


def nan_metrics():
    return {
        "RMSE": np.nan,
        "MAE": np.nan,
        "MAPE": np.nan,
        "Accuracy": np.nan,
        "Bias": np.nan,
        "ErrorPct": np.nan,
        "ErrorStd": np.nan,
    }


for model_name, spec in model_catalog.items():
    model_metrics_rows = []
    model_prediction_rows = []
    latest_fitted_model = None
    latest_test_end = None
    split_idx_global = 0

    for lookback_days in LOOKBACK_WINDOWS_DAYS:
        splits = generate_expanding_splits(
            n_rows=len(daily_df),
            initial_train_days=lookback_days,
            horizon_days=PREDICTION_WINDOW_DAYS,
            step_days=EXPAND_STEP_DAYS,
            include_forced_final=INCLUDE_FORCED_FINAL_WINDOW,
        )

        for window_id, sp in enumerate(splits, start=1):
            split_idx_global += 1

            tr_s = sp["train_start_idx"]
            tr_e = sp["train_end_idx"]
            te_s = sp["test_start_idx"]
            te_e = sp["test_end_idx"]

            train_dates = dates[tr_s:tr_e + 1]
            train_vals = values[tr_s:tr_e + 1]
            test_dates = dates[te_s:te_e + 1]
            test_vals = values[te_s:te_e + 1]

            status = "ok"
            err_msg = ""
            fitted_obj = None
            preds = np.full(len(test_vals), np.nan)

            train_metrics = nan_metrics()
            train_eval_size = 0
            train_eval_start = str(pd.Timestamp(train_dates[0]).date())
            train_eval_end = str(pd.Timestamp(train_dates[-1]).date())

            try:
                fam = spec["family"]

                if fam == "feature":
                    est = spec["builder"]()
                    X_train, y_train = build_train_xy(train_dates, train_vals, LAG_DAYS)
                    if len(X_train) < 10:
                        raise RuntimeError("not enough post-lag training rows")

                    est.fit(X_train, y_train)
                    train_pred = np.maximum(np.asarray(est.predict(X_train), dtype=float), 0.0)
                    train_metrics = metrics_dict(y_train, train_pred)

                    train_eval_size = int(len(y_train))
                    lag_max = max(LAG_DAYS)
                    train_eval_start = str(pd.Timestamp(train_dates[lag_max]).date())
                    train_eval_end = str(pd.Timestamp(train_dates[-1]).date())

                    preds = recursive_predict_feature_model(
                        estimator=est,
                        train_dates=train_dates,
                        train_vals=train_vals,
                        horizon_days=len(test_vals),
                        lag_days=LAG_DAYS,
                    )
                    fitted_obj = est

                elif fam == "arima":
                    fit = ARIMA(train_vals, order=spec["order"]).fit()
                    preds = np.maximum(np.asarray(fit.forecast(steps=len(test_vals)), dtype=float), 0.0)

                    fitted_train = np.asarray(fit.fittedvalues, dtype=float)
                    actual_train = np.asarray(train_vals[-len(fitted_train):], dtype=float)
                    fitted_train = np.maximum(fitted_train, 0.0)
                    mask = np.isfinite(actual_train) & np.isfinite(fitted_train)
                    if int(mask.sum()) >= 5:
                        train_metrics = metrics_dict(actual_train[mask], fitted_train[mask])
                    train_eval_size = int(mask.sum())

                    fitted_obj = fit

                elif fam == "sarima":
                    fit = SARIMAX(
                        train_vals,
                        order=spec["order"],
                        seasonal_order=spec["seasonal_order"],
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    ).fit(disp=False)

                    preds = np.maximum(np.asarray(fit.get_forecast(steps=len(test_vals)).predicted_mean, dtype=float), 0.0)

                    fitted_train = np.asarray(fit.fittedvalues, dtype=float)
                    actual_train = np.asarray(train_vals[-len(fitted_train):], dtype=float)
                    fitted_train = np.maximum(fitted_train, 0.0)
                    mask = np.isfinite(actual_train) & np.isfinite(fitted_train)
                    if int(mask.sum()) >= 5:
                        train_metrics = metrics_dict(actual_train[mask], fitted_train[mask])
                    train_eval_size = int(mask.sum())

                    fitted_obj = fit

                elif fam == "prophet":
                    tdf = pd.DataFrame({"ds": pd.to_datetime(train_dates), "y": np.asarray(train_vals, dtype=float)})
                    pm = Prophet(**spec["params"])
                    pm.fit(tdf)

                    in_sample = pm.predict(tdf[["ds"]])["yhat"].to_numpy(dtype=float)
                    in_sample = np.maximum(in_sample, 0.0)
                    train_metrics = metrics_dict(np.asarray(train_vals, dtype=float), in_sample)
                    train_eval_size = int(len(train_vals))

                    fut = pm.make_future_dataframe(periods=len(test_vals), freq="D")
                    fc = pm.predict(fut).tail(len(test_vals))
                    preds = np.maximum(np.asarray(fc["yhat"].to_numpy(), dtype=float), 0.0)
                    fitted_obj = pm

                else:
                    raise RuntimeError(f"unknown model family: {fam}")

            except Exception as e:
                status = "error"
                err_msg = f"{type(e).__name__}: {e}"
                preds = np.full(len(test_vals), np.nan)
                train_metrics = nan_metrics()

            if np.isnan(preds).any():
                test_metrics = nan_metrics()
            else:
                test_metrics = metrics_dict(test_vals, preds)

            if pd.notna(test_metrics["RMSE"]) and pd.notna(train_metrics["RMSE"]):
                gap_rmse = float(test_metrics["RMSE"] - train_metrics["RMSE"])
            else:
                gap_rmse = np.nan

            row = {
                "model": model_name,
                "lookback_days": int(lookback_days),
                "window_id": int(window_id),
                "split_idx_global": int(split_idx_global),
                "status": status,
                "error_message": err_msg,
                "train_start": str(pd.Timestamp(train_dates[0]).date()),
                "train_end": str(pd.Timestamp(train_dates[-1]).date()),
                "test_start": str(pd.Timestamp(test_dates[0]).date()),
                "test_end": str(pd.Timestamp(test_dates[-1]).date()),
                "n_train": int(len(train_vals)),
                "n_test": int(len(test_vals)),
                "n_train_eval": int(train_eval_size),
                "train_eval_start": str(train_eval_start),
                "train_eval_end": str(train_eval_end),
                "generalization_gap_RMSE": gap_rmse,
            }
            row.update(test_metrics)
            row.update({f"train_{k}": v for k, v in train_metrics.items()})
            model_metrics_rows.append(row)

            for dte, act, prd in zip(test_dates, test_vals, preds):
                err = act - prd if pd.notna(prd) else np.nan
                ape = (abs(err) / abs(act) * 100.0) if (pd.notna(prd) and act != 0) else np.nan
                model_prediction_rows.append(
                    {
                        "model": model_name,
                        "lookback_days": int(lookback_days),
                        "window_id": int(window_id),
                        "split_idx_global": int(split_idx_global),
                        "train_start": row["train_start"],
                        "train_end": row["train_end"],
                        "test_start": row["test_start"],
                        "test_end": row["test_end"],
                        "date": str(pd.Timestamp(dte).date()),
                        "actual": float(act),
                        "prediction": float(prd) if pd.notna(prd) else np.nan,
                        "error": float(err) if pd.notna(err) else np.nan,
                        "abs_error": float(abs(err)) if pd.notna(err) else np.nan,
                        "ape": float(ape) if pd.notna(ape) else np.nan,
                    }
                )

            if status == "ok":
                cur_end = pd.Timestamp(test_dates[-1])
                if latest_test_end is None or cur_end > latest_test_end:
                    latest_test_end = cur_end
                    latest_fitted_model = fitted_obj

    model_metrics_df = pd.DataFrame(model_metrics_rows)
    model_predictions_df = pd.DataFrame(model_prediction_rows)

    safe = sanitize_name(model_name)
    metrics_path = METRICS_DIR / f"{safe}_metrics.csv"
    preds_path = PREDICTIONS_DIR / f"{safe}_predictions.csv"
    model_path = MODELS_DIR / f"{safe}_final_model.pkl"
    meta_path = MODELS_DIR / f"{safe}_final_model_meta.json"

    model_metrics_df.to_csv(metrics_path, index=False)
    model_predictions_df.to_csv(preds_path, index=False)

    if latest_fitted_model is not None:
        with open(model_path, "wb") as f:
            cloudpickle.dump(latest_fitted_model, f)
        meta = {
            "model": model_name,
            "latest_test_end": str(latest_test_end.date()) if latest_test_end is not None else None,
            "lookbacks": LOOKBACK_WINDOWS_DAYS,
            "horizon_days": PREDICTION_WINDOW_DAYS,
            "step_days": EXPAND_STEP_DAYS,
        }
        meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    else:
        model_path = None
        meta_path = None

    artifact_manifest[model_name] = {
        "metrics_path": str(metrics_path),
        "preds_path": str(preds_path),
        "model_path": str(model_path) if model_path else None,
        "meta_path": str(meta_path) if meta_path else None,
    }

    all_metrics_rows.extend(model_metrics_rows)
    all_prediction_rows.extend(model_prediction_rows)
    print(f"Done model: {model_name} | windows: {len(model_metrics_rows)}")


all_metrics_df = pd.DataFrame(all_metrics_rows)
all_predictions_df = pd.DataFrame(all_prediction_rows)

all_metrics_path = SUMMARY_DIR / "all_models_window_metrics.csv"
all_preds_path = SUMMARY_DIR / "all_models_predictions.csv"
all_metrics_df.to_csv(all_metrics_path, index=False)
all_predictions_df.to_csv(all_preds_path, index=False)

print("Saved:", all_metrics_path)
print("Saved:", all_preds_path)

In [ ]:
# =========================
# SUMMARY: latest vs average over time
# =========================

ok_metrics = all_metrics_df[all_metrics_df["status"] == "ok"].copy()
if ok_metrics.empty:
    raise RuntimeError("No successful runs found")

ok_metrics["test_end_dt"] = pd.to_datetime(ok_metrics["test_end"])

metric_cols = ["RMSE", "MAE", "MAPE", "Accuracy", "Bias", "ErrorPct", "ErrorStd"]
for c in metric_cols:
    ok_metrics[c] = pd.to_numeric(ok_metrics[c], errors="coerce")

avg_df = ok_metrics.groupby("model", as_index=False)[metric_cols].mean()
avg_df = avg_df.rename(columns={c: f"avg_{c}" for c in metric_cols})

idx_latest = ok_metrics.sort_values("test_end_dt").groupby("model")["test_end_dt"].idxmax()
latest_df = ok_metrics.loc[idx_latest, ["model", "lookback_days", "window_id", "test_end"] + metric_cols].copy()
latest_df = latest_df.rename(columns={c: f"latest_{c}" for c in metric_cols})

slope_rows = []
for model_name, g in ok_metrics.sort_values("test_end_dt").groupby("model"):
    gg = g.reset_index(drop=True)
    if len(gg) >= 2:
        x = np.arange(len(gg), dtype=float)
        y = gg["RMSE"].to_numpy(dtype=float)
        slope = float(np.polyfit(x, y, deg=1)[0])
    else:
        slope = np.nan
    slope_rows.append({"model": model_name, "rmse_trend_slope": slope})
slope_df = pd.DataFrame(slope_rows)

summary_df = avg_df.merge(latest_df, on="model", how="left").merge(slope_df, on="model", how="left")
summary_df["latest_rmse_rank"] = summary_df["latest_RMSE"].rank(method="dense")
summary_df["avg_rmse_rank"] = summary_df["avg_RMSE"].rank(method="dense")
summary_df["final_score"] = summary_df["latest_rmse_rank"] + SELECTION_WEIGHT_AVG * summary_df["avg_rmse_rank"]
summary_df = summary_df.sort_values(["final_score", "latest_RMSE", "avg_RMSE"]).reset_index(drop=True)

best_model = summary_df.iloc[0]["model"]

lookback_summary_df = (
    ok_metrics.groupby(["model", "lookback_days"], as_index=False)[metric_cols]
    .mean()
    .sort_values(["model", "lookback_days"])
)

summary_path = SUMMARY_DIR / "model_summary_over_time.csv"
lookback_summary_path = SUMMARY_DIR / "model_lookback_summary.csv"

summary_df.to_csv(summary_path, index=False)
lookback_summary_df.to_csv(lookback_summary_path, index=False)

print("Best model candidate:", best_model)
print("Saved:", summary_path)
print("Saved:", lookback_summary_path)
summary_df.head(20)

In [ ]:
# =========================
# MLFLOW LOGGING
# parent per model, nested per window
# plus explicit final expanded nested run
# =========================

if ENABLE_MLFLOW and MLFLOW_AVAILABLE:
    import tempfile

    mlflow.set_tracking_uri(TRACKING_URI)
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = exp.experiment_id if exp is not None else mlflow.create_experiment(EXPERIMENT_NAME)

    def clean_metrics(d):
        out = {}
        for k, v in d.items():
            try:
                fv = float(v)
            except Exception:
                continue
            if np.isnan(fv) or np.isinf(fv):
                continue
            out[k] = fv
        return out

    ok_metrics = all_metrics_df[all_metrics_df["status"] == "ok"].copy()
    ok_metrics["test_end_dt"] = pd.to_datetime(ok_metrics["test_end"])

    for model_name, g in ok_metrics.groupby("model"):
        g = g.sort_values(["split_idx_global", "test_end_dt", "lookback_days", "window_id"]).reset_index(drop=True)
        latest = g.sort_values("test_end_dt").iloc[-1].to_dict()
        avg = g[["RMSE", "MAE", "MAPE", "Accuracy", "Bias", "ErrorPct", "ErrorStd"]].mean().to_dict()

        with mlflow.start_run(experiment_id=experiment_id, run_name=f"{model_name}_parent"):
            # parent run identity + setup
            mlflow.set_tags({
                "run_type": "parent",
                "model_name": model_name,
                "cv_scheme": "expanding_window",
                "window_scheme": f"lookbacks_{'-'.join(str(x) for x in LOOKBACK_WINDOWS_DAYS)}_h{PREDICTION_WINDOW_DAYS}_s{EXPAND_STEP_DAYS}",
                "feature_version": "lag_roll_v1",
            })

            mlflow.log_param("model", model_name)
            mlflow.log_param("lookbacks", ",".join(str(x) for x in LOOKBACK_WINDOWS_DAYS))
            mlflow.log_param("horizon_days", PREDICTION_WINDOW_DAYS)
            mlflow.log_param("step_days", EXPAND_STEP_DAYS)
            mlflow.log_param("lag_days", ",".join(str(x) for x in LAG_DAYS))

            # parent aggregate metrics
            parent_metrics = {f"avg_{k}": v for k, v in avg.items()}
            parent_metrics.update({f"latest_{k}": latest[k] for k in ["RMSE", "MAE", "MAPE", "Accuracy", "Bias", "ErrorPct", "ErrorStd"]})
            mlflow.log_metrics(clean_metrics(parent_metrics))

            # stepped metric history on parent
            for _, r in g.iterrows():
                step_idx = int(r["split_idx_global"])

                mlflow.log_metric("test_rmse", float(r["RMSE"]), step=step_idx)
                mlflow.log_metric("test_mae", float(r["MAE"]), step=step_idx)
                if pd.notna(r["MAPE"]):
                    mlflow.log_metric("test_mape", float(r["MAPE"]), step=step_idx)
                if pd.notna(r["Accuracy"]):
                    mlflow.log_metric("test_accuracy", float(r["Accuracy"]), step=step_idx)
                if pd.notna(r["Bias"]):
                    mlflow.log_metric("test_bias", float(r["Bias"]), step=step_idx)
                if pd.notna(r["ErrorPct"]):
                    mlflow.log_metric("test_error_pct", float(r["ErrorPct"]), step=step_idx)
                if pd.notna(r["ErrorStd"]):
                    mlflow.log_metric("test_error_std", float(r["ErrorStd"]), step=step_idx)

                if pd.notna(r["train_RMSE"]):
                    mlflow.log_metric("train_rmse", float(r["train_RMSE"]), step=step_idx)
                if pd.notna(r["train_MAE"]):
                    mlflow.log_metric("train_mae", float(r["train_MAE"]), step=step_idx)
                if pd.notna(r["train_MAPE"]):
                    mlflow.log_metric("train_mape", float(r["train_MAPE"]), step=step_idx)
                if pd.notna(r["train_Accuracy"]):
                    mlflow.log_metric("train_accuracy", float(r["train_Accuracy"]), step=step_idx)
                if pd.notna(r["train_Bias"]):
                    mlflow.log_metric("train_bias", float(r["train_Bias"]), step=step_idx)
                if pd.notna(r["train_ErrorPct"]):
                    mlflow.log_metric("train_error_pct", float(r["train_ErrorPct"]), step=step_idx)
                if pd.notna(r["train_ErrorStd"]):
                    mlflow.log_metric("train_error_std", float(r["train_ErrorStd"]), step=step_idx)

                if pd.notna(r["generalization_gap_RMSE"]):
                    mlflow.log_metric("generalization_gap_rmse", float(r["generalization_gap_RMSE"]), step=step_idx)

                mlflow.log_metric("train_window_size", float(r["n_train"]), step=step_idx)
                mlflow.log_metric("test_window_size", float(r["n_test"]), step=step_idx)

            # explicit final expanded nested run
            with mlflow.start_run(run_name=f"{model_name}_final_expanded", nested=True):
                mlflow.set_tags({"run_type": "child", "run_role": "final_expanded", "model_name": model_name})
                mlflow.log_param("lookback_days", int(latest["lookback_days"]))
                mlflow.log_param("window_id", int(latest["window_id"]))
                mlflow.log_param("split_idx_global", int(latest["split_idx_global"]))
                mlflow.log_param("train_start", str(latest["train_start"]))
                mlflow.log_param("train_end", str(latest["train_end"]))
                mlflow.log_param("test_start", str(latest["test_start"]))
                mlflow.log_param("test_end", str(latest["test_end"]))
                mlflow.log_metrics(
                    clean_metrics(
                        {
                            "test_rmse": latest["RMSE"],
                            "test_mae": latest["MAE"],
                            "test_mape": latest["MAPE"],
                            "test_accuracy": latest["Accuracy"],
                            "test_bias": latest["Bias"],
                            "test_error_pct": latest["ErrorPct"],
                            "test_error_std": latest["ErrorStd"],
                            "train_rmse": latest["train_RMSE"],
                            "train_mae": latest["train_MAE"],
                            "train_mape": latest["train_MAPE"],
                            "train_accuracy": latest["train_Accuracy"],
                            "train_bias": latest["train_Bias"],
                            "train_error_pct": latest["train_ErrorPct"],
                            "train_error_std": latest["train_ErrorStd"],
                            "generalization_gap_rmse": latest["generalization_gap_RMSE"],
                        }
                    )
                )

            # child run per split with split-level artifacts
            for _, r in g.iterrows():
                split_idx = int(r["split_idx_global"])
                run_name = f"split_{split_idx:03d}_lb{int(r['lookback_days'])}_w{int(r['window_id'])}"

                with mlflow.start_run(run_name=run_name, nested=True):
                    mlflow.set_tags({
                        "run_type": "child",
                        "run_role": "window",
                        "model_name": model_name,
                    })

                    mlflow.log_params(
                        {
                            "model": model_name,
                            "split_idx": split_idx,
                            "lookback_days": int(r["lookback_days"]),
                            "window_id": int(r["window_id"]),
                            "train_start": str(r["train_start"]),
                            "train_end": str(r["train_end"]),
                            "test_start": str(r["test_start"]),
                            "test_end": str(r["test_end"]),
                            "train_size": int(r["n_train"]),
                            "test_size": int(r["n_test"]),
                        }
                    )

                    mlflow.log_metrics(
                        clean_metrics(
                            {
                                "train_rmse": r["train_RMSE"],
                                "train_mae": r["train_MAE"],
                                "train_mape": r["train_MAPE"],
                                "train_accuracy": r["train_Accuracy"],
                                "train_bias": r["train_Bias"],
                                "train_error_pct": r["train_ErrorPct"],
                                "train_error_std": r["train_ErrorStd"],
                                "test_rmse": r["RMSE"],
                                "test_mae": r["MAE"],
                                "test_mape": r["MAPE"],
                                "test_accuracy": r["Accuracy"],
                                "test_bias": r["Bias"],
                                "test_error_pct": r["ErrorPct"],
                                "test_error_std": r["ErrorStd"],
                                "generalization_gap_rmse": r["generalization_gap_RMSE"],
                            }
                        )
                    )

                    child_preds = all_predictions_df[
                        (all_predictions_df["model"] == model_name)
                        & (all_predictions_df["split_idx_global"] == split_idx)
                    ].copy()

                    with tempfile.TemporaryDirectory(prefix="mlflow_split_") as td:
                        td_path = Path(td)
                        preds_csv = td_path / "predictions.csv"
                        residuals_csv = td_path / "residuals.csv"
                        child_preds.to_csv(preds_csv, index=False)

                        res_df = child_preds[["date", "actual", "prediction", "error", "abs_error", "ape"]].copy()
                        res_df.to_csv(residuals_csv, index=False)

                        mlflow.log_artifact(str(preds_csv), artifact_path="split_artifacts")
                        mlflow.log_artifact(str(residuals_csv), artifact_path="split_artifacts")

            # parent artifacts
            paths = artifact_manifest.get(model_name, {})
            for _, p in paths.items():
                if p and Path(p).exists():
                    mlflow.log_artifact(str(p), artifact_path=sanitize_name(model_name))

    print("MLflow logging complete")
else:
    print("MLflow skipped (disabled or package missing)")

In [ ]:
# =========================
# PLOTS + FINAL VIEW
# =========================

ok_metrics = all_metrics_df[all_metrics_df["status"] == "ok"].copy()
ok_metrics["test_end_dt"] = pd.to_datetime(ok_metrics["test_end"])

if SHOW_PLOTS and not ok_metrics.empty:
    top_models = summary_df["model"].head(TOP_MODELS_TO_PLOT).tolist()

    plt.figure(figsize=(12, 5))
    for m in top_models:
        g = ok_metrics[ok_metrics["model"] == m].sort_values("test_end_dt")
        plt.plot(g["test_end_dt"], g["RMSE"], marker="o", label=m)
    plt.title("RMSE Over Time (Expanding Windows)")
    plt.xlabel("Test End Date")
    plt.ylabel("RMSE")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

print("Deployment candidate model:", best_model)
summary_df[["model", "latest_RMSE", "avg_RMSE", "rmse_trend_slope", "final_score"]].head(20)